# Exercise 2.1: Basic Neural Network Training with PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/exercise_2_1_basic_neural_network_tabular.ipynb)

This exercise follows the first deep learning lecture. The goal is to train a small neural network on a real tabular dataset.

We use the UCI Wine Recognition dataset, a classic tutorial dataset for classification. Each row describes a wine sample using chemical measurements. The model predicts which of three cultivars the wine belongs to.

You will practice:

- downloading a small real tabular dataset
- splitting and scaling features
- converting data to PyTorch tensors
- building a neural network with `torch.nn.Module`
- training with `CrossEntropyLoss`, backpropagation, and an optimizer
- evaluating accuracy, loss curves, and a confusion matrix


## 0. Colab setup

Run this first in Colab. PyTorch is often preinstalled, but the command below makes the notebook self-contained.


In [ ]:
!pip -q install torch scikit-learn pandas matplotlib seaborn

In [ ]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## 1. Download a real tutorial dataset

The UCI Wine Recognition dataset contains chemical analysis results for wines grown in the same region in Italy. The task is to predict the cultivar class from 13 numeric measurements.

Dataset source: UCI Machine Learning Repository.


In [ ]:
wine_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"

wine_columns = [
    "class_label",
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280_od315_of_diluted_wines",
    "proline",
]

wine_df = pd.read_csv(wine_url, header=None, names=wine_columns)
wine_df["cultivar"] = "cultivar " + wine_df["class_label"].astype(str)

feature_columns = wine_columns[1:]

display(wine_df.head())
print("Rows, columns:", wine_df.shape)
print(wine_df["cultivar"].value_counts().sort_index())


In [ ]:
selected_features = [
    "alcohol",
    "malic_acid",
    "flavanoids",
    "color_intensity",
    "hue",
    "proline",
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()
for ax, col in zip(axes, selected_features):
    sns.kdeplot(data=wine_df, x=col, hue="cultivar", fill=False, common_norm=False, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


### Student task 1: Inspect the data

Before training the neural network, inspect the feature plots:

- Which features separate the cultivars clearly?
- Which cultivars look similar?
- Which feature would be hard to use alone?

<details>
<summary>Answer / tip</summary>

`flavanoids`, `color_intensity`, and `proline` often show useful separation. Some features overlap strongly, so one feature alone is usually not enough.

A neural network can combine several weak signals. That is why we train on all 13 features, not only the six features shown in the plots.
</details>


## 2. Split, scale, and convert to tensors

Neural networks train more smoothly when numerical features are scaled. We use `StandardScaler` so each feature has roughly mean 0 and standard deviation 1.

`CrossEntropyLoss` expects class labels as integer IDs: `0`, `1`, `2`, ...


In [ ]:
X = wine_df[feature_columns].values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(wine_df["cultivar"])
class_names = label_encoder.classes_

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=SEED,
    stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_ds = TensorDataset(X_val_tensor, y_val_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

print("Classes:", list(class_names))
print("Feature count:", len(feature_columns))
print("Train / val / test:", len(train_ds), len(val_ds), len(test_ds))
print("One batch shapes:")
xb, yb = next(iter(train_loader))
print("X:", xb.shape, "y:", yb.shape)


## 3. Define a small neural network

This model is a multi-layer perceptron (MLP):

1. a linear layer transforms the input features into hidden activations
2. `ReLU` adds non-linearity
3. another linear layer produces one score per class

The output values are called logits. We do not add `softmax` in the model because `CrossEntropyLoss` applies the correct transformation internally.


In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.network(x)

model = TabularMLP(
    input_dim=len(feature_columns),
    hidden_dim=16,
    output_dim=len(class_names),
).to(DEVICE)

print(model)


In [ ]:
with torch.no_grad():
    xb, yb = next(iter(train_loader))
    logits = model(xb.to(DEVICE))

print("Input batch shape: ", xb.shape)
print("Output logits shape:", logits.shape)
print("First row of logits:", logits[0].cpu().numpy().round(3))


## 4. Train the network

The training loop follows the lecture summary:

1. forward pass: compute predictions
2. loss: compare predictions with labels
3. backprop: compute gradients with `loss.backward()`
4. update: change parameters with `optimizer.step()`

PyTorch handles the gradient calculations. You only need to write the training steps in the right order.


In [ ]:
def evaluate(model, data_loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = loss_fn(logits, yb)

            total_loss += loss.item() * len(xb)
            predictions = logits.argmax(dim=1)
            correct += (predictions == yb).sum().item()
            total += len(xb)

    return total_loss / total, correct / total

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

history = []
EPOCHS = 120

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * len(xb)
        correct_train += (logits.argmax(dim=1) == yb).sum().item()
        total_train += len(xb)

    train_loss = total_train_loss / total_train
    train_acc = correct_train / total_train
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
        }
    )

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"epoch {epoch:03d} | "
            f"train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
            f"train acc {train_acc:.3f} | val acc {val_acc:.3f}"
        )

history_df = pd.DataFrame(history)


## 5. Plot the learning curves

The loss curve shows whether training is still improving. The validation curve helps identify overfitting.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=history_df, x="epoch", y="train_loss", label="train", ax=axes[0])
sns.lineplot(data=history_df, x="epoch", y="val_loss", label="validation", ax=axes[0])
axes[0].set_title("Loss during training")
axes[0].set_ylabel("Cross-entropy loss")

sns.lineplot(data=history_df, x="epoch", y="train_accuracy", label="train", ax=axes[1])
sns.lineplot(data=history_df, x="epoch", y="val_accuracy", label="validation", ax=axes[1])
axes[1].set_title("Accuracy during training")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()


### Student task 2: Change training settings

Change one setting and rerun the training cells:

- `hidden_dim`: try `4`, `16`, and `64`
- `lr`: try `0.001`, `0.01`, and `0.05`
- `EPOCHS`: try `30` and `200`

What changes in the loss and validation accuracy curves?

<details>
<summary>Answer / tip</summary>

A very small learning rate often trains slowly. A very large learning rate can make validation loss unstable. A larger hidden layer can fit the training data more easily, but on this small dataset it can also overfit.

For a quick check, compare the validation accuracy and validation loss, not only the training accuracy.
</details>


## 6. Evaluate on the test set

Use the test set only after you have chosen your model settings. This gives a more honest estimate of performance on unseen data.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"Test loss:     {test_loss:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

model.eval()
all_logits = []
with torch.no_grad():
    for xb, _ in test_loader:
        all_logits.append(model(xb.to(DEVICE)).cpu())

logits_test = torch.cat(all_logits, dim=0)
y_pred = logits_test.argmax(dim=1).numpy()

print(classification_report(y_test, y_pred, target_names=class_names))

fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=class_names,
    xticks_rotation=35,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Neural network confusion matrix")
plt.tight_layout()
plt.show()


## 7. Predict one wine sample

A trained model can be used for new tabular observations. The important detail is that the same scaler must be used before prediction.

Here we take one row from the dataset only to demonstrate the prediction workflow.


In [ ]:
new_wine = wine_df[feature_columns].iloc[[0]].copy()
true_label = wine_df["cultivar"].iloc[0]

new_scaled = scaler.transform(new_wine[feature_columns])
new_tensor = torch.tensor(new_scaled, dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    logits = model(new_tensor)
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]

prediction_table = pd.DataFrame(
    {
        "cultivar": class_names,
        "probability": probabilities,
    }
).sort_values("probability", ascending=False)

print("True label:", true_label)
display(new_wine)
display(prediction_table)


## 8. Short reflection

Answer these questions in two or three sentences each:

- What are the inputs, hidden layers, output logits, loss, and optimizer in this notebook?
- Why do we scale tabular features before training?
- Why do we keep a validation set separate from the test set?
- Why can a neural network work on tabular data even though it is often associated with images?

The point is not to memorize PyTorch syntax. The point is to understand the training workflow: forward pass, loss, backpropagation, and parameter update.
